# 📊 Notebook 02 — Exploratory Data Analysis (EDA)
## Spatio-Temporal Air Quality Analysis — Karachi, Pakistan

**Objectives:**
1. **Temporal Patterns:** Analyze seasonal, monthly, and weekly cycles of pollutants.
2. **Spatial Distribution:** Compare pollution levels across the 8 monitoring stations (Industrial vs Residential).
3. **Meteorological Correlation:** Study the impact of wind speed, humidity, and temperature on air quality.
4. **Proxy Relationships:** Validate S5P Aerosol Index against MODIS AOD.
5. **Holiday Impact:** Investigate pollution dips during Eid and Ramadan.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from statsmodels.tsa.seasonal import seasonal_decompose
from pathlib import Path

# Set plotting style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('viridis')
pd.options.display.max_columns = None

# ── Load Merged Dataset ───────────────────────────────────────────────────────
DATA_PATH = 'data/processed/merged_karachi_dataset.csv'

if not Path(DATA_PATH).exists():
    print(f'⚠️ Dataset not found at {DATA_PATH}')
    print('Please ensure Notebook 01 has been run and exports are merged.')
else:
    df = pd.read_csv(DATA_PATH, parse_dates=['date'])
    print(f'✓ Loaded dataset: {df.shape}')
    display(df.head())

## 1. Temporal Analysis

In [ ]:
# Rolling average of Aerosol Index (PM Proxy)
if 'df' in locals():
    plt.figure(figsize=(15, 6))
    sns.lineplot(data=df, x='date', y='aer_ai', hue='station', alpha=0.3)
    df_citywide = df.groupby('date')['aer_ai'].mean().rolling(30).mean()
    plt.plot(df_citywide.index, df_citywide.values, color='black', linewidth=2, label='30-Day Citywide Moving Avg')
    plt.title('Karachi Aerosol Index (S5P) Time Series (2019-2024)', fontsize=15)
    plt.ylabel('Absorbing Aerosol Index')
    plt.legend()
    plt.show()

## 2. Seasonal & Cyclical Patterns

In [ ]:
if 'df' in locals():
    df['month'] = df['date'].dt.month
    df['day_name'] = df['date'].dt.day_name()
    
    fig, ax = plt.subplots(1, 2, figsize=(18, 6))
    
    # Monthly seasonality
    sns.boxplot(data=df, x='month', y='aer_ai', ax=ax[0])
    ax[0].set_title('Monthly Seasonality (PM Proxy)')
    
    # Weekly cycle
    day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
    sns.boxplot(data=df, x='day_name', y='aer_ai', order=day_order, ax=ax[1])
    ax[1].set_title('Weekly Cycle (Traffic Impact)')
    plt.xticks(rotation=45)
    
    plt.show()

## 3. Spatial Comparison

In [ ]:
if 'df' in locals():
    plt.figure(figsize=(12, 6))
    station_ranks = df.groupby('station')['aer_ai'].median().sort_values(ascending=False).index
    sns.barplot(data=df, x='station', y='aer_ai', order=station_ranks, palette='magma')
    plt.title('Pollution Ranking by Station (Median Aerosol Index)')
    plt.xticks(rotation=45)
    plt.show()

## 4. Meteorological Influence

In [ ]:
if 'df' in locals():
    # Select meteorological and pollutant columns
    meteo_cols = ['temperature_2m', 'wind_speed', 'rh', 'total_precipitation_sum', 'aer_ai', 'no2', 'so2', 'co']
    corr = df[meteo_cols].corr()
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f', center=0)
    plt.title('Correlation Matrix: Pollutants vs Meteorology')
    plt.show()

## 5. Holiday Impact (Ramadan/Eid)

In [ ]:
if 'df' in locals() and 'is_ramadan' in df.columns:
    plt.figure(figsize=(10, 6))
    sns.boxplot(data=df, x='is_ramadan', y='aer_ai')
    plt.title('Aerosol Index: Normal Days vs Ramadan')
    plt.show()
    
    if 'is_eid' in df.columns:
        plt.figure(figsize=(10, 6))
        sns.boxplot(data=df, x='is_eid', y='aer_ai')
        plt.title('Aerosol Index: Normal Days vs Eid Holidays')
        plt.show()